In [2]:
import json
import datetime

In [3]:
def two_hours_forward(timestamp_string: str):
    date, time = timestamp_string.split(' ')
    year, month, day = date.split('-')
    hour, minute, second = time.split(':')
    date = datetime.datetime(int(year), int(month), int(day), int(hour), int(minute), int(second))
    date += datetime.timedelta(hours=2)
    return date

In [4]:
def get_weekday(timestamp_string: str):
    date, time = timestamp_string.split(' ')
    year, month, day = date.split('-')
    hour, minute, second = time.split('.')
    weekday = datetime.datetime(int(year), int(month), int(day), int(hour), int(minute), int(second)).weekday()
    return weekday

In [5]:
def time_to_integer(timestamp_string: str):
    output = 0
    date, time = timestamp_string.split(' ')
    hour, minute, _ = time.split('.')
    hour, minute = int(hour), int(minute)
    output += hour
    return output

In [6]:
# Loading txt_files from json
filename = '2024-july-aug-veturilo.json'
# filename = '2024-08-18to25-veturilo.json'

# Decide if you need station/freestanding txt_files (important, if you use the file with txt_files from whole year, you are likely to run out of memory
FREESTANDING = True
STATIONS = False
WEATHER_DATA = False

# Dictionary representing all stations and freestanding bikes for every 5 minutes
timestamp_dictionary = dict()

# Extracting stations info for stations in Warsaw
with open(f'json_files/bike_data/{filename}', 'r', encoding='utf8') as file:
    for line in file:
        data_piece = json.loads(line)
        if FREESTANDING:
            if data_piece['bike']:
                if data_piece['city_uid'] == 812:
                    timestamp = str(two_hours_forward(data_piece['timestamp']['$date'].replace('T', ' ')[:-1])).replace(':', '.')
                    if not timestamp in timestamp_dictionary.keys():
                        timestamp_dictionary[timestamp] = []
                    timestamp_dictionary[timestamp].append(data_piece)
        if STATIONS:
            if not data_piece['bike']:
                if data_piece['city_uid'] == 812:
                    timestamp = str(two_hours_forward(data_piece['timestamp']['$date'].replace('T', ' ')[:-1])).replace(':', '.')
                    if not timestamp in timestamp_dictionary.keys():
                        timestamp_dictionary[timestamp] = []
                    timestamp_dictionary[timestamp].append(data_piece)     
                    
weather_details = []
if WEATHER_DATA:
    with open('weather_data.json') as json_file:
        weather_details = json.load(json_file)

In [7]:
# Preparing text files
target_folder = "txt_files"

# Write in data with all data
ALL_DATA = True
# Create separate files for every weekday
WEEKDAY_SPLITTING = True
# Data from Monday-Thursday will be stored in single file
MON_THU_FRI_SAT_SUN = False

if MON_THU_FRI_SAT_SUN and not WEEKDAY_SPLITTING:
    raise Exception("Enable weekday splitting to split txt_files merge Mon-Thu data")

# Decide what kind of data you want to include
LATITUDE = True
LONGITUDE = True
TIME = True
AVERAGE_TEMPERATURE = False
RAIN = False

BOUNDARY_WEST = 20.9
BOUNDARY_EAST = 21.2
BOUNDARY_SOUTH = 52.05
BOUNDARY_NORTH = 52.35

attributes_monday = open(f"{target_folder}/monday_data.txt", "w")
attributes_tuesday = open(f"{target_folder}/tuesday_data.txt", "w")
attributes_wednesday = open(f"{target_folder}/wednesday_data.txt", "w")
attributes_thursday = open(f"{target_folder}/thursday_data.txt", "w")

attributes_monday_thursday = open(f"{target_folder}/monday_thursday_data.txt", "w")

attributes_friday = open(f"{target_folder}/friday_data.txt", "w")
attributes_saturday = open(f"{target_folder}/saturday_data.txt", "w")
attributes_sunday = open(f"{target_folder}/sunday_data.txt", "w")

attributes_all = open(f"{target_folder}/all_data.txt", "w")


for timestamp in timestamp_dictionary.keys():
    for bike_stand in timestamp_dictionary[timestamp]:
        # Filtering essential regions data
        if BOUNDARY_WEST < bike_stand['lng'] < BOUNDARY_EAST and BOUNDARY_SOUTH < bike_stand['lat'] < BOUNDARY_NORTH and bike_stand['bike']:
            single_attribute = []
            if LONGITUDE:
                single_attribute.append(bike_stand['lng'])
            if LATITUDE:
                single_attribute.append(bike_stand['lat'])
            if TIME:
                single_attribute.append(time_to_integer(timestamp))
            if AVERAGE_TEMPERATURE:
                single_attribute.append(weather_details[timestamp[:10]]['avg_temperature'])
            if RAIN:
                single_attribute.append(weather_details[timestamp[:10]]['rain'])
            for _ in range(bike_stand['bikes']):
                if WEEKDAY_SPLITTING:
                    if get_weekday(timestamp) < 4:
                        if MON_THU_FRI_SAT_SUN:
                            attributes_monday_thursday.write(str(single_attribute)[1:-1] + '\n')
                        else:
                            if get_weekday(timestamp) == 0:
                                attributes_monday.write(str(single_attribute)[1:-1] + '\n')
                            elif get_weekday(timestamp) == 1:
                                attributes_tuesday.write(str(single_attribute)[1:-1] + '\n')
                            elif get_weekday(timestamp) == 2:
                                attributes_wednesday.write(str(single_attribute)[1:-1] + '\n')
                            else:
                                attributes_thursday.write(str(single_attribute)[1:-1] + '\n')
                    elif get_weekday(timestamp) == 4:
                        attributes_friday.write(str(single_attribute)[1:-1] + '\n')
                    elif get_weekday(timestamp) == 5:
                        attributes_saturday.write(str(single_attribute)[1:-1] + '\n')
                    else:
                        attributes_sunday.write(str(single_attribute)[1:-1] + '\n')
                if ALL_DATA:
                    attributes_all.write(str(single_attribute)[1:-1] + '\n')

attributes_monday.close()
attributes_tuesday.close()
attributes_wednesday.close()
attributes_thursday.close()

attributes_monday_thursday.close()

attributes_friday.close()
attributes_saturday.close()
attributes_sunday.close()

attributes_all.close()